In [ ]:
import numpy as np
import pandas as pd

def clean_data(df):

    # Drop column: 'Title', 'firstname', 'nickname', 'lastname', 'birthdate', ... 'Path'
    df = df.drop(columns=['Title', 'firstname', 'nickname', 'lastname', 'birthdate', 'referral_narrative', 'Item Type2', 'Path3', 'Item Type', 'Path'])

    # Fill NaN values in 'patient_apx_age' with 0 before type casting
    df['patient_apx_age'] = df['patient_apx_age'].fillna(0).astype('int64')

    # Replace patient_age > 103 with median
    median_age = df['patient_age'].median().astype('int64')
    df['patient_age'] = df['patient_age'].apply(lambda x: median_age if x > 103 else x)

    # Replace missing values in column: 'patient_age' with values from column: 'patient_apx_age' and drop column: 'patient_apx_age'
    df['patient_age'] = df['patient_age'].replace('', pd.NA).fillna(df['patient_apx_age'])
    df = df.drop(columns=['patient_apx_age'])

    # Replace long labels and fill missing values in column: 'patient_zipcode' with 'Homeless'
    df['patient_zipcode'] = df['patient_zipcode'].replace(['', 'Experiencing Homelessness, no current ZIP Code', 'Non-Clallam County ZIP Code'], pd.NA).fillna('Homeless')

    # Change column type to string for column: 'patient_sex' and 'patient_zipcode'
    df = df.astype({'patient_sex': 'string', 'patient_zipcode': 'string'})

    # Replace Zip Codes in column: "patient_zipcode" with text labels
    zip_code_di = {"98363": "PA West", "98362": "PA East", "98331": "Forks", "98382": "Sequim", "98364": "Homeless", "98381": "Sekiu", "98365": "Homeless", "98386": "Homeless", "98357": "Neah Bay", "Other": "Homeless"}
    df['patient_zipcode'] = df['patient_zipcode'].replace(zip_code_di)


    # Replace missing values with "Uninsured" in column: 'patient_insurance'
    df = df.fillna({'patient_insurance': 'Uninsured'})
    df = df.replace({'MedicaidOther': 'Medicaid', 'Medicaid, Other': 'Medicaid', 'Private, Medicaid': 'Medicaid'})
    df['patient_insurance'] = df['patient_insurance'].astype('string')

    # Change column type to string for column: 'living_situation'
    df['living_situation'] = df['living_situation'].fillna('No data')
    df = df.astype({'living_situation': 'string'})

    # Fill referral_time missing values with 0000
    df['referral_time'] = df['referral_time'].fillna(0).astype('int64').apply(lambda x: f"{x:04}")

    # Replace value in row 92 of referral_time with 0736
    df.at[91, 'referral_time'] = '0736'

    # Fill referral_time missing values with 0000
    df['overdose_time'] = df['overdose_time'].fillna(0).astype('int64').apply(lambda x: f"{x:04}")

    # Replace missing values with the most common value of each column in: 'referral_date'
    df = df.fillna({'referral_date': df['referral_date'].mode()[0]})

    # Replace referral_date time with referral_time time if referral_date time is 00:00:00
    df['referral_date'] = df.apply(
        lambda row: row['referral_date'].replace(
            hour=int(row['referral_time'][:2]),
            minute=int(row['referral_time'][2:4]),
            second=0
        ) if row['referral_date'].time() == pd.Timestamp('00:00:00').time() else row['referral_date'], axis=1)

    # Replace missing overdose_date with Created
    df["overdose_date"] = df["overdose_date"].fillna(df['Created'])

    # Replace referral_date time with referral_time time if referral_date time is 00:00:00
    df['overdose_date'] = df.apply(
        lambda row: row['overdose_date'].replace(
            hour=int(row['overdose_time'][:2]),
            minute=int(row['overdose_time'][2:4]),
            second=0
        ) if row['overdose_date'].time() == pd.Timestamp('00:00:00').time() else row['overdose_date'], axis=1)

    # Drop columns: 'referral_time', 'overdose_time'
    df = df.drop(columns=['referral_time', 'overdose_time'])

    # Change column type to string for column: 'delay_in_referral'
    # Calculate the difference in hours between referral_date and overdose_date
    # Define the conditions
    # Define the corresponding values
    # Apply the conditions and values
    # Drop the temporary column
    # Change column type to string for column: 'result'
    # Drop column: 'delay_in_referral'
    # Rename column 'result' to 'delay_in_referral'
    df = df.astype({'delay_in_referral': 'string'})
    df['time_diff_hours'] = (df['referral_date'] - df['overdose_date']).dt.total_seconds() / 3600
    conditions = [
        (df['referral_date'] == df['overdose_date']),
        (df['time_diff_hours'] < 24),
        (df['time_diff_hours'] >= 24) & (df['time_diff_hours'] < 72),
        (df['time_diff_hours'] >= 72) & (df['time_diff_hours'] < 168),
        (df['time_diff_hours'] >= 168) & (df['time_diff_hours'] < 672),
        (df['time_diff_hours'] >= 672),
        (df['cpm_disposition'] == 'Cancelled')
    ]
    values = [
        'CPM responded',
        '< 24hrs',
        '24-72hrs',
        '3-7 days',
        '1-4 weeks',
        '> 1 month',
        'Call cancelled'
    ]
    df['result'] = np.select(conditions, values, default='Other')
    df.drop(columns=['time_diff_hours'], inplace=True)
    df = df.astype({'result': 'string'})
    df = df.drop(columns=['delay_in_referral'])
    df = df.rename(columns={'result': 'delay_in_referral'})

    # Replace missing values with "" in column: 'cpm_notification'
    df = df.fillna({'cpm_notification': "No data"})

    # Change column type to string for column: 'cpm_notification'
    df = df.astype({'cpm_notification': 'string'})

    # Replace missing values with "No data" in column: 'cpm_disposition'
    df = df.fillna({'cpm_disposition': "No data"})

    # Change column type to string for column: 'cpm_disposition'
    df = df.astype({'cpm_disposition': 'string'})

    # Drop columns: 'referral_type', 'referral_personnel' and 6 other columns
    df = df.drop(columns=['referral_type', 'referral_personnel', 'referral_agency_category', 'referral_agency', 'rp_details', 'referral_status', 'referral_closed_reason', 'referral_status_note'])

    # Replace missing values with the most common value of each column in: 'referral_source'
    df = df.fillna({'referral_source': df['referral_source'].mode()[0]})

    # Replace all instances of "Other" with "CPM Research" in column: 'referral_source'
    df['referral_source'] = df['referral_source'].str.replace("Other", "CPM Research", case=False, regex=False)

    # Change column type to string for column: 'referral_source'
    df = df.astype({'referral_source': 'string'})

    # Drop columns: 'encounter_type_cat1', 'encounter_type_cat2'
    df = df.drop(columns=['encounter_type_cat1', 'encounter_type_cat2'])

    # Replace missing values with "" in column: 'encounter_type_cat3'
    df = df.fillna({'encounter_type_cat3': "No data"})

    # Drop column: 'encounter_type_cat3'
    df = df.drop(columns=['encounter_type_cat3'])

    # Drop column: 'incident_number'
    df = df.drop(columns=['incident_number'])

    # Fill missing or 'Unknown' od_address with default value
    df['od_address'] = df['od_address'].replace('Unknown', '2321 W 18th St')
    df['od_address'] = df['od_address'].fillna('2321 W 18th St')

    # Replace all instances of ", Port Angeles, WA 98362" with "" in column: 'od_address'
    df['od_address'] = df['od_address'].str.replace(", Port Angeles, WA 98362", "", case=False, regex=False)

    # Change column type to string for column: 'od_address'
    df = df.astype({'od_address': 'string'})

    # Replace missing values with the most common value of each column in: 'engagement_location'
    df = df.fillna({'engagement_location': df['engagement_location'].mode()[0]})

    # Change column type to string for column: 'engagement_location'
    df = df.astype({'engagement_location': 'string'})

    # Replace 'Unknown' with 'No data' in number_of_nonems_onscene
    df['number_of_nonems_onscene'] = df['number_of_nonems_onscene'].replace('Unknown', 'No data')
    df['number_of_nonems_onscene'] = df['number_of_nonems_onscene'].fillna('No data')

    # Replace 'Unknown' with 'No data' in number_of_nonems_onscene
    df['number_of_ems_onscene'] = df['number_of_ems_onscene'].replace('Unknown', 'No data')
    df['number_of_ems_onscene'] = df['number_of_ems_onscene'].fillna('No data')

    # Replace 'Unknown' with 'No data' in number_of_nonems_onscene
    df['number_of_peers_onscene'] = df['number_of_peers_onscene'].replace('Unknown', 'No data')
    df['number_of_peers_onscene'] = df['number_of_peers_onscene'].fillna('No data')

    # Replace 'Unknown' with 'No data' in number_of_police_onscene
    df['number_of_police_onscene'] = df['number_of_police_onscene'].replace('Unknown', 'No data')
    df['number_of_police_onscene'] = df['number_of_police_onscene'].fillna('No data')

    df = df.fillna(
        {
            "suspected_drug": "No data",
            "cpr_administered": "No data",
            "police_ita": "No data",
            "disposition": "No data",
            "transport_to_location": "No data",
            "transported_by": "No data"
        }
    )
    df['suspected_drug'] = df['suspected_drug'].str.replace(";#", ", ", case=False, regex=False)

    # Replace specific values in suspected_drug
    df['suspected_drug'] = df['suspected_drug'].replace({
        'Opiate - unk/other': 'Opiate - Other',
        'Methamphetamine': 'Meth',
        'Sedative - unk/other': 'Sedative'
    })

    # Split text using string ',' in column: 'suspected_drug'
    loc_0 = df.columns.get_loc('suspected_drug')
    df_split = df['suspected_drug'].str.split(pat=',', expand=True).add_prefix('suspected_drug_')
    df = pd.concat([df.iloc[:, :loc_0], df_split, df.iloc[:, loc_0:]], axis=1)
    df = df.drop(columns=['suspected_drug'])

    # Append ', Polypharmacy' to suspected_drug_0 if suspected_drug_1 has a value
    df.loc[df['suspected_drug_1'].notna(), 'suspected_drug_0'] += ', Polypharmacy'

    # Drop column: 'suspected_drug_1'
    df = df.drop(columns=['suspected_drug_1'])

    # Change column type to string for column: 'suspected_drug_0'
    df = df.astype({'suspected_drug_0': 'string'})

    # Rename column 'suspected_drug_0' to 'suspected_drug'
    df = df.rename(columns={'suspected_drug_0': 'suspected_drug'})

    # Replace all instances of "Unknown" with "No CPR" in column: 'cpr_administered'
    df['cpr_administered'] = df['cpr_administered'].str.replace("Unknown", "No CPR", case=False, regex=False)

    # Change column type to string for column: 'cpr_administered'
    df = df.astype({'cpr_administered': 'string'})

    # Replace all instances of "" with "" in column: 'police_ita'
    df['police_ita'] = df['police_ita'].str.replace("Unknown", "No data", case=False, regex=False)

    # Change column type to string for column: 'police_ita'
    df = df.astype({'police_ita': 'string'})

    # Change column type to string for column: 'disposition'
    df = df.astype({'disposition': 'string'})

    # Change column type to string for column: 'transport_to_location'
    df = df.astype({'transport_to_location': 'string'})

    # Change column type to string for column: 'transported_by'
    df = df.astype({'transported_by': 'string'})

    # Replace missing values with "0" in column: 'narcan_doses_prior_to_ems'
    df = df.fillna({'narcan_doses_prior_to_ems': "0"})

    # Replace all instances of "None Given" with "0" in column: 'narcan_doses_prior_to_ems'
    df['narcan_doses_prior_to_ems'] = df['narcan_doses_prior_to_ems'].str.replace("None Given", "0", case=False, regex=False)

    # Change column type to string for column: 'narcan_doses_prior_to_ems'
    df = df.astype({'narcan_doses_prior_to_ems': 'string'})
    df['narcan_prior_to_ems_dosage'] = df['narcan_prior_to_ems_dosage'].fillna(0).astype('int64')
    df['narcan_doses_by_ems'] = df['narcan_doses_by_ems'].fillna(0)
    df['narcan_doses_by_ems'] = df['narcan_doses_by_ems'].replace('None Given', '0').astype('int64')
    df['narcan_doses_prior_to_ems'] = df['narcan_doses_prior_to_ems'].replace('> 6', '6').astype('int64')
    df['narcan_by_ems_dosage'] = df['narcan_by_ems_dosage'].fillna(0)

    # Drop column: 'supplies_provided'
    df = df.drop(columns=['supplies_provided'])

    df['leave_behind_narcan_amount'] = df['leave_behind_narcan_amount'].fillna(0).astype('int64')

    # Drop column: 'harmreduce_supplies_type'
    df = df.drop(columns=['harmreduce_supplies_type'])

    # Drop column: 'persons_trained'
    df = df.drop(columns=['persons_trained'])

    # Replace missing values with 0 in column: 'referral_to_sud_agency'
    df = df.fillna({'referral_to_sud_agency': 0})

    # Change column type to int64 for column: 'referral_to_sud_agency'
    df = df.astype({'referral_to_sud_agency': 'int64'})

    # Replace missing values with "Patient referred" in column: 'no_referral_reason'
    df = df.fillna({'no_referral_reason': "Patient referred"})

    # Change column type to string for column: 'no_referral_reason'
    df = df.astype({'no_referral_reason': 'string'})

    # Replace missing values with 0 in column: 'referral_rediscovery'
    df = df.fillna({'referral_rediscovery': 0})

    # Change column type to int64 for column: 'referral_rediscovery'
    df = df.astype({'referral_rediscovery': 'int64'})

    # Replace missing values with 0 in column: 'referral_reflections'
    df = df.fillna({'referral_reflections': 0})

    # Change column type to int64 for column: 'referral_reflections'
    df = df.astype({'referral_reflections': 'int64'})

    # Replace missing values with 0 in column: 'referral_pbh'
    df = df.fillna({'referral_pbh': 0})

    # Change column type to int64 for column: 'referral_pbh'
    df = df.astype({'referral_pbh': 'int64'})

    # Replace missing values with 0 in column: 'referral_other'
    df = df.fillna({'referral_other': 0})

    # Change column type to int64 for column: 'referral_other'
    df = df.astype({'referral_other': 'int64'})

    # Replace missing values with "No data" in columns: 'contact_level_rediscovery', 'contact_level_reflections' and 2 other columns
    df = df.fillna({'contact_level_rediscovery': "No data", 'contact_level_reflections': "No data", 'contact_level_pbh': "No data", 'contact_level_other': "No data"})

    # Replace missing values with 0 in columns: 'accepted_rediscovery', 'accepted_reflections' and 2 other columns
    df = df.fillna({'accepted_rediscovery': 0, 'accepted_reflections': 0, 'accepted_pbh': 0, 'accepted_other': 0})

    # Change column type to int64 for columns: 'accepted_rediscovery', 'accepted_reflections' and 2 other columns
    df = df.astype({'accepted_rediscovery': 'int64', 'accepted_reflections': 'int64', 'accepted_pbh': 'int64', 'accepted_other': 'int64'})

    # Change column type to string for columns: 'contact_level_rediscovery', 'contact_level_reflections' and 2 other columns
    df = df.astype({'contact_level_rediscovery': 'string', 'contact_level_reflections': 'string', 'contact_level_pbh': 'string', 'contact_level_other': 'string'})

    # Drop columns: 'no_accepting_agency_reason', 'real_team_roi'
    df = df.drop(columns=['no_accepting_agency_reason', 'real_team_roi'])

    # Replace missing values with 0 in column: 'is_bup_indicated'
    df = df.fillna({'is_bup_indicated': 0})

    # Change column type to int64 for column: 'is_bup_indicated'
    df = df.astype({'is_bup_indicated': 'int64'})

    # Replace missing values with "No data" in column: 'bup_not_indicated_reason'
    df = df.fillna({'bup_not_indicated_reason': "No data"})

    # Change column type to string for column: 'bup_not_indicated_reason'
    df = df.astype({'bup_not_indicated_reason': 'string'})

    # Replace missing values with "No data" in column: 'bup_already_prescribed'
    df = df.fillna({'bup_already_prescribed': "No data"})

    # Replace all instances of "Unknown" with "No data" in column: 'bup_already_prescribed'
    df['bup_already_prescribed'] = df['bup_already_prescribed'].str.replace("Unknown", "No data", case=False, regex=False)

    # Change column type to string for column: 'bup_already_prescribed'
    df = df.astype({'bup_already_prescribed': 'string'})

    # Replace missing values with 0 in column: 'bup_admin'
    df = df.fillna({'bup_admin': 0})

    # Change column type to int64 for column: 'bup_admin'
    df = df.astype({'bup_admin': 'int64'})

    # Replace missing values with 0 in column: 'client_agrees_to_mat'
    df = df.fillna({'client_agrees_to_mat': 0})

    # Change column type to int64 for column: 'client_agrees_to_mat'
    df = df.astype({'client_agrees_to_mat': 'int64'})

    # Drop columns: 'cows1time', 'cows1' and 4 other columns
    df = df.drop(columns=['cows1time', 'cows1', 'cows2time', 'cows2', 'cows3time', 'cows3'])

    # Drop columns: 'bup_doses_admin', 'bup_doses_dosage'
    df = df.drop(columns=['bup_doses_admin', 'bup_doses_dosage'])

    # Drop columns: 'referral_to_wd_management', 'referral_to_wm_agency', 'wm_accepting_agency'
    df = df.drop(columns=['referral_to_wd_management', 'referral_to_wm_agency', 'wm_accepting_agency'])

    # Drop column: 'withdrawals_only'
    df = df.drop(columns=['withdrawals_only'])

    # Drop columns: 'overdose_recent', 'withdrawals_from_narcan' and 5 other columns
    df = df.drop(columns=['overdose_recent', 'withdrawals_from_narcan', 'last_opiate_use_days', 'hours_withdrawing', 'Created', 'Modified', 'ref_status'])

    # Replace all instances of "No data" with "0" in columns: 'number_of_nonems_onscene', 'number_of_ems_onscene' and 2 other columns
    df['number_of_nonems_onscene'] = df['number_of_nonems_onscene'].str.replace("No data", "0", case=False, regex=False).astype('int64')
    df['number_of_ems_onscene'] = df['number_of_ems_onscene'].str.replace("No data", "0", case=False, regex=False).astype('int64')
    df['number_of_peers_onscene'] = df['number_of_peers_onscene'].str.replace("No data", "0", case=False, regex=False).astype('int64')
    df['number_of_police_onscene'] = df['number_of_police_onscene'].str.replace("No data", "0", case=False, regex=False).astype('int64')

    return df

# Loaded variable 'df' from file referrals_port.xlsx
df = pd.read_excel('referrals_port.xlsx')

df_clean = clean_data(df.copy())

# Save clean data to a new csv file
df_clean.to_csv('referrals_port_clean.csv', index=False)